# GeoXGB vs XGBoost — Best Performance on train.csv

**Core idea:** HVRT selects the most geometrically diverse representative samples
from the 630k train.csv pool. Models are trained *only* on that reduced set.
The ~126k holdout gives high-confidence AUC estimates.

**Pipeline:**
1. Load `train.csv` (630k CTGAN samples)
2. Stratified 80/20 holdout split → ~504k train / ~126k holdout (fixed)
3. HVRT reduce ~504k training samples to `HVRT_N_KEEP` representative samples
4. 5-fold CV on retained samples: each fold trains on ~80%, evaluates on the holdout
5. HPO via multiprocessing — GeoXGB (108 configs) + XGBoost (36 configs) × 5 folds
6. Select best config per model by mean holdout AUC across 5 folds
7. Final models trained on HVRT-retained set, evaluated on holdout
8. XGBoost baseline: same best config trained on full ~504k training set (reference)

**Note on `HVRT_N_KEEP`:** At ~504k training samples, even 1% gives 5,040 samples.
`HVRT_N_KEEP=5000` keeps the HPO at similar speed to the previous experiment.
GeoXGB's internal `auto_expand` grows each fold's ~4,000 seeds to ~5,000 synthetic
samples per iteration. Increase `HVRT_N_KEEP` for richer retained-data fidelity
at the cost of longer HPO wall time.

In [2]:
import multiprocessing
import time
import warnings
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


In [3]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

RANDOM_STATE = 42
HOLDOUT_FRAC = 0.20
N_FOLDS      = 5

# HVRT absolute target: number of representative samples to retain from the
# ~504k training pool.  HVRT_N_KEEP=5000 ≈ ~1% of the training data and keeps
# HPO at a similar speed to the previous experiment.
# Increase for higher retained-data fidelity (slower HPO).
HVRT_N_KEEP  = 5000

DATA_PATH  = "../data/train.csv"
LABEL_COL  = "Heart Disease"

# Fixed GeoXGB params
GEO_FIXED = dict(
    n_rounds         = 1000,
    auto_noise       = True,
    cache_geometry   = False,
    auto_expand      = True,
    min_samples_leaf = 5,
    random_state     = RANDOM_STATE,
)

# Fixed XGBoost params
XGB_FIXED = dict(
    n_estimators = 1000,
    random_state = RANDOM_STATE,
    n_jobs       = 1,
    eval_metric  = "logloss",
)

# GeoXGB HPO grid: 3 x 4 x 3 x 3 = 108 configs
GEO_GRID = dict(
    max_depth      = [3, 4, 5],
    learning_rate  = [0.10, 0.15, 0.20, 0.25],
    reduce_ratio   = [0.6, 0.7, 0.8],
    refit_interval = [10, 15, 20],
)

# XGBoost HPO grid: 3 x 4 x 3 = 36 configs
XGB_GRID = dict(
    max_depth     = [3, 4, 5],
    learning_rate = [0.10, 0.15, 0.20, 0.25],
    subsample     = [0.7, 0.8, 1.0],
)

def _build_configs(grid):
    keys, values = list(grid.keys()), list(grid.values())
    return [dict(zip(keys, combo)) for combo in product(*values)]

geo_configs = _build_configs(GEO_GRID)
xgb_configs = _build_configs(XGB_GRID)

print(f"GeoXGB configs : {len(geo_configs)}")
print(f"XGBoost configs: {len(xgb_configs)}")
print(f"Total HPO jobs : {(len(geo_configs) + len(xgb_configs)) * N_FOLDS}  "
      f"({N_FOLDS} folds × {len(geo_configs)+len(xgb_configs)} configs)")
print(f"CPU count      : {multiprocessing.cpu_count()}")

GeoXGB configs : 108
XGBoost configs: 36
Total HPO jobs : 720  (5 folds × 144 configs)
CPU count      : 32


In [4]:
# ---------------------------------------------------------------------------
# 1. Load train.csv — stratified 80/20 holdout split
# ---------------------------------------------------------------------------

print(f"Loading {DATA_PATH} ...")
t0 = time.perf_counter()
df = pd.read_csv(DATA_PATH)
print(f"  Loaded {len(df):,} rows in {time.perf_counter()-t0:.1f}s")

feature_cols = [c for c in df.columns if c != LABEL_COL]
X = df[feature_cols].values.astype(np.float64)
y = LabelEncoder().fit_transform(df[LABEL_COL])

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y,
    test_size    = HOLDOUT_FRAC,
    random_state = RANDOM_STATE,
    stratify     = y,
)

ct = np.bincount(y_train)
ch = np.bincount(y_holdout)
print(f"\nDataset     : {len(y):,} samples, {X.shape[1]} features")
print(f"Training    : {len(y_train):,}  "
      f"(Absence={ct[0]:,}, Presence={ct[1]:,}, "
      f"{ct[1]/len(y_train)*100:.1f}% Presence)")
print(f"Holdout     : {len(y_holdout):,}  "
      f"(Absence={ch[0]:,}, Presence={ch[1]:,})")

Loading ../data/train.csv ...
  Loaded 630,000 rows in 0.4s

Dataset     : 630,000 samples, 14 features
Training    : 504,000  (Absence=278,037, Presence=225,963, 44.8% Presence)
Holdout     : 126,000  (Absence=69,509, Presence=56,491)


In [5]:
# ---------------------------------------------------------------------------
# 2. HVRT reduce training set to HVRT_N_KEEP representative samples
# ---------------------------------------------------------------------------

from hvrt import HVRT

print(f"Fitting HVRT on {len(X_train):,} training samples...")
t0 = time.perf_counter()
h_pre = HVRT(random_state=RANDOM_STATE)   # auto-tune min_samples_leaf (fine at 504k)
h_pre.fit(X_train, y_train.astype(float))
print(f"  HVRT.fit   : {time.perf_counter()-t0:.1f}s  "
      f"({len(h_pre.unique_partitions_)} partitions)")

print(f"Selecting {HVRT_N_KEEP:,} samples via FPS...")
t0 = time.perf_counter()
X_red, red_idx = h_pre.reduce(
    n                 = HVRT_N_KEEP,
    method            = "fps",
    variance_weighted = True,
    return_indices    = True,
)
y_red = y_train[red_idx]
print(f"  HVRT.reduce: {time.perf_counter()-t0:.1f}s")

cr = np.bincount(y_red)
print(f"\nRetained : {len(X_red):,} samples  "
      f"(Absence={cr[0]:,}, Presence={cr[1]:,}, "
      f"{cr[1]/len(y_red)*100:.1f}% Presence)")
print(f"Fraction : {len(X_red)/len(X_train)*100:.3f}% of {len(X_train):,} training samples")

Fitting HVRT on 504,000 training samples...
  HVRT.fit   : 2.1s  (535 partitions)
Selecting 5,000 samples via FPS...
  HVRT.reduce: 1.4s

Retained : 5,000 samples  (Absence=2,327, Presence=2,673, 53.5% Presence)
Fraction : 0.992% of 504,000 training samples


In [6]:
# ---------------------------------------------------------------------------
# 3. Build 5-fold CV splits on the retained samples
# ---------------------------------------------------------------------------

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_splits = list(skf.split(X_red, y_red))

print(f"5-fold CV on {len(X_red):,}-sample retained set:")
for fi, (tr, te) in enumerate(fold_splits):
    ctr = np.bincount(y_red[tr])
    print(f"  Fold {fi}: train={len(tr):,}  "
          f"(Absence={ctr[0]:,}, Presence={ctr[1]:,})  "
          f"→ evaluated on holdout ({len(y_holdout):,} samples)")

5-fold CV on 5,000-sample retained set:
  Fold 0: train=4,000  (Absence=1,862, Presence=2,138)  → evaluated on holdout (126,000 samples)
  Fold 1: train=4,000  (Absence=1,862, Presence=2,138)  → evaluated on holdout (126,000 samples)
  Fold 2: train=4,000  (Absence=1,862, Presence=2,138)  → evaluated on holdout (126,000 samples)
  Fold 3: train=4,000  (Absence=1,861, Presence=2,139)  → evaluated on holdout (126,000 samples)
  Fold 4: train=4,000  (Absence=1,861, Presence=2,139)  → evaluated on holdout (126,000 samples)


In [7]:
# ---------------------------------------------------------------------------
# 4. CV worker  (module-level for Windows/loky pickling)
# ---------------------------------------------------------------------------

def _cv_worker(model_type, config, X_tr, y_tr, X_holdout, y_holdout,
               geo_fixed, xgb_fixed):
    """
    Fit one (model_type, config) on X_tr, return holdout AUC.
    model_type : 'geo' | 'xgb'
    """
    import warnings as _w; _w.filterwarnings("ignore")
    from sklearn.metrics import roc_auc_score

    try:
        if model_type == "geo":
            from geoxgb import GeoXGBClassifier
            params = {**geo_fixed, **config}
            clf = GeoXGBClassifier(**params)
            clf.fit(X_tr, y_tr)
            proba = clf.predict_proba(X_holdout)[:, 1]

        else:  # 'xgb'
            from xgboost import XGBClassifier
            params = {**xgb_fixed, **config}
            clf = XGBClassifier(**params)
            clf.fit(X_tr, y_tr)
            proba = clf.predict_proba(X_holdout)[:, 1]

        return float(roc_auc_score(y_holdout, proba))

    except Exception:
        return 0.0

print("Worker defined.")

Worker defined.


In [8]:
# ---------------------------------------------------------------------------
# 5. Build job list and run HPO in parallel
#    Job = (model_type, config, fold_train_X, fold_train_y)
# ---------------------------------------------------------------------------

import time

# job_keys: (model_type, config_idx, fold_idx) for aggregation
jobs     = []
job_keys = []

for fi, (tr_idx, _) in enumerate(fold_splits):
    X_tr_fold = X_red[tr_idx]
    y_tr_fold = y_red[tr_idx]

    for ci, cfg in enumerate(geo_configs):
        jobs.append(("geo", cfg, X_tr_fold, y_tr_fold,
                     X_holdout, y_holdout, GEO_FIXED, XGB_FIXED))
        job_keys.append(("geo", ci, fi))

    for ci, cfg in enumerate(xgb_configs):
        jobs.append(("xgb", cfg, X_tr_fold, y_tr_fold,
                     X_holdout, y_holdout, GEO_FIXED, XGB_FIXED))
        job_keys.append(("xgb", ci, fi))

n_jobs_total = len(jobs)
print(f"Launching {n_jobs_total} jobs across {multiprocessing.cpu_count()} workers...")
print(f"  GeoXGB: {len(geo_configs)} configs × {N_FOLDS} folds = "
      f"{len(geo_configs)*N_FOLDS} jobs")
print(f"  XGBoost: {len(xgb_configs)} configs × {N_FOLDS} folds = "
      f"{len(xgb_configs)*N_FOLDS} jobs")

t0 = time.perf_counter()
raw_aucs = Parallel(
    n_jobs  = multiprocessing.cpu_count(),
    prefer  = "processes",
    verbose = 5,
)(
    delayed(_cv_worker)(*j) for j in jobs
)
wall = time.perf_counter() - t0
print(f"\nDone. Wall time: {wall:.1f}s")

Launching 720 jobs across 32 workers...
  GeoXGB: 108 configs × 5 folds = 540 jobs
  XGBoost: 36 configs × 5 folds = 180 jobs


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   8 tasks      | elapsed:   49.0s
[Parallel(n_jobs=32)]: Done  98 tasks      | elapsed:  3.3min
[Parallel(n_jobs=32)]: Done 224 tasks      | elapsed:  7.4min
[Parallel(n_jobs=32)]: Done 386 tasks      | elapsed: 11.8min
[Parallel(n_jobs=32)]: Done 584 tasks      | elapsed: 18.3min



Done. Wall time: 1382.8s


[Parallel(n_jobs=32)]: Done 720 out of 720 | elapsed: 23.0min finished


In [ ]:
# ---------------------------------------------------------------------------
# 6. Aggregate: mean holdout AUC across 5 folds per (model_type, config)
# ---------------------------------------------------------------------------

from collections import defaultdict

fold_aucs = defaultdict(list)  # key = (model_type, config_idx)
for (mt, ci, fi), auc in zip(job_keys, raw_aucs):
    fold_aucs[(mt, ci)].append(auc)

# Mean and std across folds
geo_scores = [
    (geo_configs[ci], np.mean(aucs), np.std(aucs))
    for (mt, ci), aucs in fold_aucs.items() if mt == "geo"
]
xgb_scores = [
    (xgb_configs[ci], np.mean(aucs), np.std(aucs))
    for (mt, ci), aucs in fold_aucs.items() if mt == "xgb"
]

geo_scores.sort(key=lambda x: -x[1])
xgb_scores.sort(key=lambda x: -x[1])

best_geo_cfg, best_geo_auc, best_geo_std = geo_scores[0]
best_xgb_cfg, best_xgb_auc, best_xgb_std = xgb_scores[0]

# ---- GeoXGB table ----
print("GeoXGB — Top 10 configs (mean holdout AUC across 5 folds)")
print(f"  {'rank':>4}  {'mean AUC':>9}  {'std':>6}  "
      f"{'depth':>5}  {'lr':>5}  {'rr':>5}  {'ri':>4}")
print(f"  {'----':>4}  {'---------':>9}  {'------':>6}  "
      f"{'-----':>5}  {'-----':>5}  {'-----':>5}  {'----':>4}")
for rank, (cfg, mean_auc, std_auc) in enumerate(geo_scores[:10], 1):
    print(
        f"  {rank:>4d}  {mean_auc:>9.5f}  {std_auc:>6.4f}  "
        f"{cfg['max_depth']:>5d}  {cfg['learning_rate']:>5.2f}  "
        f"{cfg['reduce_ratio']:>5.1f}  {cfg['refit_interval']:>4d}"
    )

# ---- XGBoost table ----
print()
print("XGBoost — Top 10 configs (mean holdout AUC across 5 folds)")
print(f"  {'rank':>4}  {'mean AUC':>9}  {'std':>6}  "
      f"{'depth':>5}  {'lr':>5}  {'subsample':>9}")
print(f"  {'----':>4}  {'---------':>9}  {'------':>6}  "
      f"{'-----':>5}  {'-----':>5}  {'---------':>9}")
for rank, (cfg, mean_auc, std_auc) in enumerate(xgb_scores[:10], 1):
    print(
        f"  {rank:>4d}  {mean_auc:>9.5f}  {std_auc:>6.4f}  "
        f"{cfg['max_depth']:>5d}  {cfg['learning_rate']:>5.2f}  "
        f"{cfg['subsample']:>9.1f}"
    )

# ---- Summary ----
print()
print("Best config summary (5-fold mean):")
print(f"  GeoXGB  : {best_geo_auc:.5f} ± {best_geo_std:.4f}  {best_geo_cfg}")
print(f"  XGBoost : {best_xgb_auc:.5f} ± {best_xgb_std:.4f}  {best_xgb_cfg}")
leader = "GeoXGB" if best_geo_auc >= best_xgb_auc else "XGBoost"
print(f"  Leader  : {leader}  (+{abs(best_geo_auc - best_xgb_auc):.5f})")

In [ ]:
# ---------------------------------------------------------------------------
# 7. Final models: train on HVRT-retained set, evaluate on holdout
#    + XGBoost baseline: train on full ~504k training set for reference
# ---------------------------------------------------------------------------
from geoxgb import GeoXGBClassifier

# --- GeoXGB: best config on retained set ---
print(f"Fitting GeoXGB (best config) on {len(X_red):,} retained samples...")
t0 = time.perf_counter()
geo_final = GeoXGBClassifier(**{**GEO_FIXED, **best_geo_cfg})
geo_final.fit(X_red, y_red)
print(f"  done in {time.perf_counter()-t0:.1f}s")

# --- XGBoost: best config on retained set ---
print(f"Fitting XGBoost (best config) on {len(X_red):,} retained samples...")
t0 = time.perf_counter()
xgb_final = XGBClassifier(**{**XGB_FIXED, **best_xgb_cfg})
xgb_final.fit(X_red, y_red)
print(f"  done in {time.perf_counter()-t0:.1f}s")

# --- XGBoost baseline: best config on FULL training set ---
print(f"Fitting XGBoost baseline on full {len(X_train):,} training samples...")
t0 = time.perf_counter()
xgb_baseline = XGBClassifier(
    **{**XGB_FIXED, **best_xgb_cfg},
    n_jobs=multiprocessing.cpu_count(),
)
xgb_baseline.fit(X_train, y_train)
print(f"  done in {time.perf_counter()-t0:.1f}s")

# --- Holdout AUC ---
geo_auc      = roc_auc_score(y_holdout, geo_final.predict_proba(X_holdout)[:, 1])
xgb_auc      = roc_auc_score(y_holdout, xgb_final.predict_proba(X_holdout)[:, 1])
xgb_base_auc = roc_auc_score(y_holdout, xgb_baseline.predict_proba(X_holdout)[:, 1])

print(f"\nHoldout AUC ({len(y_holdout):,} samples):")
print(f"  GeoXGB  (HVRT {len(X_red):,}) : {geo_auc:.5f}")
print(f"  XGBoost (HVRT {len(X_red):,}) : {xgb_auc:.5f}")
print(f"  XGBoost (full {len(X_train):,}) : {xgb_base_auc:.5f}  ← baseline")
print(f"\n  GeoXGB vs XGBoost (retained) : {geo_auc - xgb_auc:+.5f}")
print(f"  XGBoost retained vs full     : {xgb_auc - xgb_base_auc:+.5f}")

In [ ]:
# ---------------------------------------------------------------------------
# 8. GeoXGB provenance and feature importances
# ---------------------------------------------------------------------------

prov = geo_final.sample_provenance()
print("GeoXGB sample provenance:")
for k, v in prov.items():
    print(f"  {k:<20s}: {v}")
print(f"  noise_estimate       : {geo_final.noise_estimate():.4f}")
print(f"  n_resamples          : {geo_final.n_resamples}")

print("\nGeoXGB feature importances (boosting):")
importances = geo_final.feature_importances(feature_names=feature_cols)
for feat, imp in importances.items():
    bar = "#" * int(imp * 40)
    print(f"  {feat:<30s} {imp:.4f}  {bar}")

## Results Summary

| Step | Detail |
|---|---|
| Dataset | train.csv — 630k CTGAN samples, 13 features |
| Holdout | 20% stratified (~126k samples, fixed seed=42) |
| HVRT retain | ~504k → 5,000 samples (FPS, auto-tuned min_samples_leaf) |
| CV | 5-fold on 5,000 retained; each fold (~4,000 samples) evaluated on ~126k holdout |
| GeoXGB HPO | 108 configs × 5 folds = 540 jobs |
| XGBoost HPO | 36 configs × 5 folds = 180 jobs |
| GeoXGB grid | max_depth [3,4,5] × lr [0.1,0.15,0.2,0.25] × reduce_ratio [0.6,0.7,0.8] × refit_interval [10,15,20] |
| XGBoost grid | max_depth [3,4,5] × lr [0.1,0.15,0.2,0.25] × subsample [0.7,0.8,1.0] |
| Fixed (GeoXGB) | n_rounds=1000, auto_noise=True, cache_geometry=False, auto_expand=True |
| Fixed (XGBoost) | n_estimators=1000 |
| Final GeoXGB | Best config trained on 5,000 retained samples |
| Final XGBoost | Best config trained on 5,000 retained samples |
| XGBoost baseline | Same best XGBoost config trained on full ~504k training set |

**Increasing `HVRT_N_KEEP`** retains more of the training pool and may improve
model quality, particularly for XGBoost which benefits more from larger training
sets. Doubling to 10,000 roughly doubles HPO wall time.